In [0]:
from pyspark.sql.functions import arrays_zip,explode,col,coalesce,when,row_number,to_json,from_json,substring,lit,get_json_object,monotonically_increasing_id
from pyspark.sql.types import IntegerType
from pyspark.sql.window import Window

In [0]:
iplData = spark.read.format('parquet').option('inferSchema',True).load('/Volumes/ipl/bronze/ipldata')

In [0]:

inningsDF = iplData.withColumn("inning", explode("innings"))
oversDF = inningsDF.withColumn("over", explode("inning.overs"))
ballsDF = oversDF.withColumn("delivery", explode("over.deliveries"))
dateDF = ballsDF.withColumn("dates", explode(col("info.dates")))

rn = Window.partitionBy(col('info.season')).orderBy(col("dates"))
cleandInningsData = dateDF.select(
    coalesce(col("info.event.match_number"),row_number().over(rn)).alias("matchNumber"),
    (substring(col("info.season"),-2,2).cast(IntegerType())+lit(2000)).alias("season"),
    col("inning.team").alias("team"),
    col("over.over").alias("overNumber"),
    # row_number().over(ballNum).alias('deliveries'),
    col("delivery.batter").alias("striker"),
    col("delivery.non_striker").alias("nonStriker"),
    col("delivery.bowler").alias("bowler"),
    col("delivery.runs.total").alias("runsTotal"),
    col("delivery.runs.batter").alias("runsBatter"),
    col("delivery.runs.extras").alias("runsExtras"),
    (when(col("delivery.extras.byes").isNotNull(),'byes') \
        .when(col("delivery.extras.legbyes").isNotNull(),'legbyes') \
        .when(col("delivery.extras.wides").isNotNull(),'wides') \
        .when(col("delivery.extras.noballs").isNotNull(),'noballs') \
        .when(col("delivery.extras.penalty").isNotNull(),'penalty') \
        .otherwise('NA')
      ).alias('extrasKind'),
    col("delivery.wickets").alias("wickets")
)
ballNum = Window.partitionBy(col('season')
                             ,col('team')
                             ,col('matchNumber')
                             ,col('overNumber')) \
                .orderBy(monotonically_increasing_id())

scoreCleanedData = cleandInningsData.withColumn('deliveries',row_number().over(ballNum))
scores = scoreCleanedData.select('matchNumber,season,overNumber,deliveries,striker,nonStriker,runsTotal,runsBatter,runsExtras,bowler,extrasKind')
# display(ScoreCleandData)
ScoreCleandData.write.format('delta').mode('overwrite').saveAsTable('ipl.silver.scores')